In [ ]:
import pandas as pd
import re
import numpy as np
import tensorflow as tlf
from tensorflow.keras.layers import TextVectorization

In [ ]:
df = pd.read_csv('data/processed_data.csv')
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming tec...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,petter mattei's love in the time of money is a...,1


In [25]:
X_data = df['review']
y = df['sentiment']
y = y.array

# EMBEDDING

In [48]:
vocab_size = 10000
max_len = 350

vectorizer_layer = TextVectorization(
    max_tokens = vocab_size,
    output_mode = 'int', 
    output_sequence_length = max_len
)

In [49]:
vectorizer_layer.adapt(X_data)
X_padded = vectorizer_layer(X_data)

len(vectorizer_layer.get_vocabulary())

10000

In [50]:
X = X_padded.numpy()

In [109]:
X

array([[  28,    5,    2, ...,    0,    0,    0],
       [   4,  384,  114, ...,    0,    0,    0],
       [  10,  193,   11, ...,    0,    0,    0],
       ...,
       [  10,  228,    4, ...,    0,    0,    0],
       [ 143,  161,    6, ...,    0,    0,    0],
       [  55,   28, 5717, ...,    0,    0,    0]], shape=(50000, 350))

In [51]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, SimpleRNN, Input

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TRAINING THE MODEL

In [75]:
model = Sequential([
    Input(shape = (max_len, )),
    Embedding(input_dim = vocab_size, output_dim = 128, mask_zero = True),
    SimpleRNN(64, activation = 'tanh'),
    Dense(1, activation = 'sigmoid')
])


In [76]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 350, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_8 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,292,417 (4.93 MB)

 Trainable params: 1,292,417 (4.93 MB)

 Non-trainable params: 0 (0.00 B)

In [77]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [78]:
from tensorflow.keras.callbacks import EarlyStopping

es_callback = EarlyStopping(monitor = 'val_loss', patience = 5, restore_best_weights = True)

In [79]:
history = model.fit(
    X_train, y_train, 
    epochs = 15, batch_size = 32,
    validation_split = 0.2,
    callbacks = [es_callback]
)

Epoch 1/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 104s 101ms/step - accuracy: 0.6883 - loss: 0.5898 - val_accuracy: 0.7081 - val_loss: 0.5580
Epoch 2/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 100s 100ms/step - accuracy: 0.7036 - loss: 0.5643 - val_accuracy: 0.6444 - val_loss: 0.6248
Epoch 3/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 98s 98ms/step - accuracy: 0.7997 - loss: 0.4422 - val_accuracy: 0.7204 - val_loss: 0.5698
Epoch 4/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 99s 99ms/step - accuracy: 0.8558 - loss: 0.3356 - val_accuracy: 0.7546 - val_loss: 0.5866
Epoch 5/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 98s 98ms/step - accuracy: 0.8928 - loss: 0.2633 - val_accuracy: 0.7103 - val_loss: 0.7441
Epoch 6/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 97s 97ms/step - accuracy: 0.8972 - loss: 0.2474 - val_accuracy: 0.6984 - val_loss: 0.7711


# PREDICTION

In [100]:
def get_review(review):
    review_vectorized = vectorizer_layer([review])

    prediction = model.predict(review_vectorized)
    
    label = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return label, float(f"{prediction[0][0]:.2f}")

In [101]:
review = "This movie was absolutely amazing. It was a masterpiece in every way"
get_review(review)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step


('Positive', 0.65)

In [102]:
review = "This movie was not what I hoped for. I didn't expected it to be this bad"
get_review(review)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


('Negative', 0.09)

In [ ]:
import os
import pickle

# saving textvectorizer
with open(os.path.join(os.getcwd(), 'models/textvectorizer.pkl'), 'wb') as file:
    pickle.dump(vectorizer_layer, file)

with open(os.path.join(os.getcwd(), 'models/SimpleRNNmodel.h5'), 'wb') as file:
    pickle.dump(model, file)